
# 練習問題: ループをGPUで並列実行する

## 目標

`#pragma omp target teams distribute parallel for` (Fortran では `!$omp target teams distribute parallel do` ... `!$omp end target teams distribute parallel do`) を1つ挿入するだけで, 1重ループをGPU上の多数のチーム×スレッドに分割して並列実行できることを体験する.

## 課題

`offload_loop.cpp` (または `offload_loop.f90`) は, 現状では `m` 回まわるループを逐次に実行し, 各繰り返しが「自分の繰り返し番号・チーム番号・スレッド番号」を表示する.
コメント `TODO` の指示に従って **OpenMP の指示行を1つ追加** し, このループをGPU上で並列に実行させよ.

- C++: `for` 文の直前に `#pragma omp target teams distribute parallel for` を1行加える.
- Fortran: `do` ループを `!$omp target teams distribute parallel do` と `!$omp end target teams distribute parallel do` で囲む.

ループは結果を配列に書き戻さず表示するだけなので, `map` 節を考える必要はない.
それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -mp=gpu offload_loop.cpp -o offload_loop.exe

# Fortran
nvfortran -mp=gpu offload_loop.f90 -o offload_loop.exe
```

GPUは計算ノードにのみ搭載されているので, `%%bash_submit` でジョブとして投入して実行する.
チーム数は `OMP_NUM_TEAMS`, 1チームあたりのスレッド数は `OMP_NUM_THREADS` で指定する.
`OMP_NUM_THREADS` は 1 または 32 の倍数でなければならないことに注意 (GPUのスレッドは32本単位(ワープ)で動くため).

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

OMP_NUM_TEAMS=2 OMP_NUM_THREADS=32 ./offload_loop.exe 8
```

## 期待される結果

並列化前(逐次)では, すべての繰り返しがチーム0・スレッド0で実行され, 表示も繰り返し番号順に並ぶ.

並列化後は, 各繰り返しが複数のチーム・スレッドに分散して実行される. 表示の順番は実行ごとに入れ替わってよい. 例 (`m=8`):

```
i = 3  executed by team 1  thread 3
i = 0  executed by team 0  thread 0
i = 5  executed by team 1  thread 5
...
```

`OMP_NUM_TEAMS` や `OMP_NUM_THREADS`, コマンドライン引数 `m` を変えて, 繰り返しがどのチーム・スレッドに割り当てられるか観察せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ offload_loop.cpp
#include <cstdio>
#include <cstdlib>
#include <omp.h>

int main(int argc, char ** argv) {
  long m = (1 < argc ? atol(argv[1]) : 8);
  // TODO: 下の for 文の直前に #pragma omp target teams distribute parallel for を1行追加し, ループをGPU上の多数のチーム×スレッドで並列実行させよ. (結果を表示するだけなので map 節は不要)
  for (long i = 0; i < m; i++) {
    printf("i = %ld  executed by team %d  thread %d\n",
           i, omp_get_team_num(), omp_get_thread_num());
  }
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=gpu offload_loop.cpp -o offload_loop_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./offload_loop_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ offload_loop.f90
program offload_loop
  use omp_lib
  implicit none
  character(len=32) :: arg
  integer(8) :: m, i

  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) m
  else
     m = 8
  end if
  ! TODO: 下の do ループを !$omp target teams distribute parallel do ... !$omp end target teams distribute parallel do で囲み, ループをGPU上の多数のチーム×スレッドで並列実行させよ. (結果を表示するだけなので map 節は不要)
  do i = 1, m
     print "(a,i0,a,i0,a,i0)", "i = ", i, "  executed by team ", &
          omp_get_team_num(), "  thread ", omp_get_thread_num()
  end do
  ! TODO: 上で始めた target teams distribute parallel do 領域を閉じる (!$omp end target teams distribute parallel do).
end program offload_loop

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=gpu offload_loop.f90 -o offload_loop_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./offload_loop_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:offload_loop.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:offload_loop.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:offload_loop.cpp}